# CS-781 Phase 5 - Legal Statute Identification (multi-label) on our own Phase-1 models
**Settings:** Accelerator **GPU T4 x2**, Internet **On**, secret **HF_TOKEN** switched ON. Then **Save Version -> Save & Run All**.

Everything is uploaded to `<you>/cs781-p1-ckpt/phase5/`; finished runs are skipped if you re-run.
1. Cell 3 - downloads the official `Phase_5_LSI` data once, cleans + tokenises it with our Phase-1 tokenizer, caches it on HF.
2. Cell 7 - 3-step safety test on the real data, then 14 experiments on both GPUs (each has a time budget; about 6-6.5 h in total):
   all four Phase-1 models; LoRMA (compulsory), full fine-tuning, partial (top-layer) fine-tuning, bottleneck adapters, prompt tuning,
   knowledge distillation (llama teacher -> shared student, and the same student without KD); reading: up to 4 chunks covering the
   whole document (attention pooling), chunk-level predictions + max, first 512, last 512; losses: BCE with per-statute positive
   weights, plain BCE, focal loss.
3. Cell 8 - results, threshold strategies (global vs per-statute, at-least-one-statute rule), rare statutes, co-occurrence, confusions,
   and a cross-check of every run with the OFFICIAL `evaluate_phase5.py` on dev.
   Predictions only use statute ids up to the highest id seen in dev: the official evaluator rejects the WHOLE file if any id is
   >= (highest id in its truth) + 1.
4. Cell 9 - ensembles (`ens_top3`, `ens_top5`, `ens_greedy`).

Dev labels are used only to pick checkpoints and thresholds - never for training.


In [1]:
# ===================== P5 CELL 1 - Setup + our tokenizer (Phase 5: Legal Statute Identification) =====================
# Settings: Accelerator = GPU T4 x2, Internet = On, Secret "HF_TOKEN" (or "HF") switched ON for this notebook.
import os, sys, json, time, math, random, re, shutil, collections, subprocess
os.environ["HF_HOME"] = "/tmp/hf_home"
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi, hf_hub_download
_sec = UserSecretsClient()
try:
    HF_TOKEN = _sec.get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = _sec.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)
HF_USER   = api.whoami()["name"]
DATA_REPO = f"{HF_USER}/cs781-p1-data"          # our tokenizer
CKPT_REPO = f"{HF_USER}/cs781-p1-ckpt"          # Phase-1 models; Phase-5 outputs go to its phase5/ folder
SRC_REPO  = "Exploration-Lab/CS-781-Capstone"
WORK      = "/kaggle/working"
P5        = "/tmp/cs781/p5"; os.makedirs(P5, exist_ok=True)
RAW       = f"{P5}/flat"; os.makedirs(RAW, exist_ok=True)
DATA      = f"{P5}/data"; os.makedirs(DATA, exist_ok=True)       # token cache (train/dev/test.npz)

# ---- our tokenizer (Phase 1) ----
TOK_PATH = hf_hub_download(DATA_REPO, "tokenizer.json", repo_type="dataset", token=HF_TOKEN, local_dir=f"{P5}/tok")
from tokenizers import Tokenizer
tok = Tokenizer.from_file(TOK_PATH)
print("tokenizer vocab:", tok.get_vocab_size())

import torch
print(f"torch {torch.__version__} | GPUs {torch.cuda.device_count()}",
      *[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


tokenizer.json:   0%|          | 0.00/2.32M [00:00<?, ?B/s]

tokenizer vocab: 32768
torch 2.11.0+cu128 | GPUs 2 Tesla T4 Tesla T4


In [2]:
%%writefile /kaggle/working/model.py
"""CS-781 Phase 1 - four legal language-model architectures, written from scratch in PyTorch.

  gpt2    : classic decoder-only Transformer (learned absolute positions, LayerNorm, GELU MLP, multi-head attention)
  llama   : modern decoder-only Transformer (RoPE, RMSNorm, SwiGLU, grouped-query attention, no biases)
  shared  : lightweight decoder (ALBERT-style factorised embeddings + cycle cross-layer weight sharing + multi-query attn)
  encdec  : encoder-decoder Transformer (bidirectional encoder over the prompt, causal decoder with cross-attention)

All models: tied input/output embeddings, pre-norm residual blocks, PyTorch SDPA attention, KV-cache generation.
"""
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class ModelConfig:
    arch: str = "llama"
    vocab_size: int = 32768
    d_model: int = 768
    n_layer: int = 12            # decoder-only: layers; encdec: layers in EACH of encoder and decoder; shared: effective depth
    n_head: int = 12
    n_kv_head: int = 4           # GQA groups (llama/encdec); shared uses 1 (MQA); gpt2 uses n_head (MHA)
    d_ff: int = 2304
    max_seq_len: int = 1024
    rope_theta: float = 10000.0
    dropout: float = 0.0
    n_unique_layers: int = 6     # shared only: number of distinct blocks, reused cyclically
    d_embed: int = 256           # shared only: factorised embedding size
    norm_eps: float = 1e-5

    def to_dict(self):
        return asdict(self)


def preset(arch: str, **overrides) -> ModelConfig:
    base = {
        "gpt2":   dict(arch="gpt2", d_model=768, n_layer=12, n_head=12, n_kv_head=12, d_ff=3072),
        "llama":  dict(arch="llama", d_model=768, n_layer=12, n_head=12, n_kv_head=4, d_ff=2304),
        "shared": dict(arch="shared", d_model=768, n_layer=12, n_head=12, n_kv_head=1, d_ff=3072,
                       n_unique_layers=6, d_embed=256),
        "encdec": dict(arch="encdec", d_model=768, n_layer=6, n_head=12, n_kv_head=4, d_ff=2304),
    }[arch]
    base.update(overrides)
    return ModelConfig(**base)


# ----------------------------------------------------------------------------------------------------------------
# building blocks
# ----------------------------------------------------------------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).to(x.dtype)


class LayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.ln = nn.LayerNorm(d, eps=eps)

    def forward(self, x):
        return F.layer_norm(x.float(), self.ln.normalized_shape, self.ln.weight.float(), self.ln.bias.float(),
                            self.ln.eps).to(x.dtype)


def rope_cache(head_dim, max_len, theta, device=None):
    inv = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(max_len, device=device).float()
    freqs = torch.outer(t, inv)                       # (T, hd/2)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, cos, sin):
    # x: (B, H, T, hd); cos/sin: (T, hd/2) - rotate pairs (x[..., :hd/2], x[..., hd/2:])
    xf = x.float()
    x1, x2 = xf[..., : xf.shape[-1] // 2], xf[..., xf.shape[-1] // 2:]
    cos, sin = cos[None, None], sin[None, None]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.to(x.dtype)


class Attention(nn.Module):
    """Multi-head / grouped-query / multi-query attention. Self-attention (optionally causal) or cross-attention."""

    def __init__(self, cfg: ModelConfig, n_kv_head: int, bias: bool, use_rope: bool, cross: bool = False):
        super().__init__()
        d, h = cfg.d_model, cfg.n_head
        assert d % h == 0 and h % n_kv_head == 0
        self.h, self.kv, self.hd = h, n_kv_head, d // h
        self.use_rope, self.cross = use_rope, cross
        self.q = nn.Linear(d, h * self.hd, bias=bias)
        self.k = nn.Linear(d, n_kv_head * self.hd, bias=bias)
        self.v = nn.Linear(d, n_kv_head * self.hd, bias=bias)
        self.o = nn.Linear(h * self.hd, d, bias=bias)
        self.o.is_residual_proj = True
        self.dropout = cfg.dropout

    def _split(self, x, n):
        B, T, _ = x.shape
        return x.view(B, T, n, self.hd).transpose(1, 2)          # (B, n, T, hd)

    def _expand(self, x):
        return x if self.kv == self.h else x.repeat_interleave(self.h // self.kv, dim=1)

    def forward(self, x, rope=None, pos=0, causal=True, cache=None, memory=None):
        """
        x: (B, T, d).  rope: (cos, sin) full tables.  pos: absolute position of x[:,0] (for RoPE during decoding).
        cache: dict used during generation. Self-attn: keys 'k','v' accumulate. Cross-attn: 'k','v' computed once.
        memory: encoder output (B, S, d) for cross-attention.
        """
        q = self._split(self.q(x), self.h)
        if self.cross:
            if cache is not None and "k" in cache:
                k, v = cache["k"], cache["v"]
            else:
                k, v = self._split(self.k(memory), self.kv), self._split(self.v(memory), self.kv)
                if cache is not None:
                    cache["k"], cache["v"] = k, v
            is_causal = False
        else:
            k, v = self._split(self.k(x), self.kv), self._split(self.v(x), self.kv)
            if self.use_rope:
                cos, sin = rope
                T = x.shape[1]
                q = apply_rope(q, cos[pos:pos + T], sin[pos:pos + T])
                k = apply_rope(k, cos[pos:pos + T], sin[pos:pos + T])
            if cache is not None:
                if "k" in cache:
                    k = torch.cat([cache["k"], k], dim=2); v = torch.cat([cache["v"], v], dim=2)
                cache["k"], cache["v"] = k, v
            # causal mask only needed when queries and keys cover the same span (training / prefill)
            is_causal = causal and q.shape[2] == k.shape[2] and q.shape[2] > 1
            if causal and q.shape[2] != k.shape[2] and q.shape[2] != 1:
                raise ValueError("multi-token decode with a non-empty cache is not supported")
        y = F.scaled_dot_product_attention(q, self._expand(k), self._expand(v), is_causal=is_causal,
                                           dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(x.shape[0], x.shape[1], self.h * self.hd)
        return self.o(y)


class MLP(nn.Module):
    def __init__(self, cfg: ModelConfig, kind: str, bias: bool):
        super().__init__()
        self.kind = kind
        if kind == "swiglu":
            self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
            self.up = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
        else:
            self.up = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=bias)
        self.down.is_residual_proj = True
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        if self.kind == "swiglu":
            h = F.silu(self.gate(x)) * self.up(x)
        else:
            h = F.gelu(self.up(x), approximate="tanh")
        return self.drop(self.down(h))


class Block(nn.Module):
    """Pre-norm Transformer block: x + Attn(Norm(x)) [+ CrossAttn(Norm(x), memory)] + MLP(Norm(x))."""

    def __init__(self, cfg: ModelConfig, style: str, n_kv_head: int, cross: bool = False):
        super().__init__()
        gpt2 = style == "gpt2"
        Norm = LayerNorm if gpt2 else RMSNorm
        self.n1 = Norm(cfg.d_model, cfg.norm_eps)
        self.attn = Attention(cfg, n_kv_head, bias=gpt2, use_rope=not gpt2)
        self.has_cross = cross
        if cross:
            self.nc = Norm(cfg.d_model, cfg.norm_eps)
            self.xattn = Attention(cfg, n_kv_head, bias=False, use_rope=False, cross=True)
        self.n2 = Norm(cfg.d_model, cfg.norm_eps)
        self.mlp = MLP(cfg, "gelu" if gpt2 else "swiglu", bias=gpt2)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, rope=None, pos=0, causal=True, cache=None, memory=None, xcache=None):
        x = x + self.drop(self.attn(self.n1(x), rope=rope, pos=pos, causal=causal, cache=cache))
        if self.has_cross:
            x = x + self.drop(self.xattn(self.nc(x), memory=memory, cache=xcache))
        return x + self.mlp(self.n2(x))


# ----------------------------------------------------------------------------------------------------------------
# the language model
# ----------------------------------------------------------------------------------------------------------------
class LegalLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        a = cfg.arch
        assert a in ("gpt2", "llama", "shared", "encdec"), a
        d = cfg.d_model
        if a == "shared":
            self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_embed)
            self.emb_proj = nn.Linear(cfg.d_embed, d, bias=False)          # E -> d   (factorised embedding)
        else:
            self.tok_emb = nn.Embedding(cfg.vocab_size, d)
        if a == "gpt2":
            self.pos_emb = nn.Embedding(cfg.max_seq_len, d)
        self.drop = nn.Dropout(cfg.dropout)

        if a == "gpt2":
            self.blocks = nn.ModuleList([Block(cfg, "gpt2", cfg.n_head) for _ in range(cfg.n_layer)])
        elif a == "llama":
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_layer)])
        elif a == "shared":
            assert cfg.n_layer % cfg.n_unique_layers == 0
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_unique_layers)])
        else:  # encdec
            self.enc_blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_layer)])
            self.enc_norm = RMSNorm(d, cfg.norm_eps)
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head, cross=True) for _ in range(cfg.n_layer)])
        self.norm = LayerNorm(d, cfg.norm_eps) if a == "gpt2" else RMSNorm(d, cfg.norm_eps)

        if a != "gpt2":
            hd = d // cfg.n_head
            cos, sin = rope_cache(hd, max(2 * cfg.max_seq_len, 2048), cfg.rope_theta)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init)
        n_res = 2 * self.n_layer_applications() + (cfg.n_layer if a == "encdec" else 0)
        for name, p in self.named_parameters():
            if name.endswith("o.weight") or name.endswith("down.weight"):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(n_res))

    # -- helpers ------------------------------------------------------------------------------------------------
    def n_layer_applications(self):
        return self.cfg.n_layer * (2 if self.cfg.arch == "encdec" else 1)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def _rope(self):
        return None if self.cfg.arch == "gpt2" else (self.rope_cos, self.rope_sin)

    def _embed(self, idx, pos=0):
        x = self.tok_emb(idx)
        if self.cfg.arch == "shared":
            x = self.emb_proj(x)
        if self.cfg.arch == "gpt2":
            T = idx.shape[1]
            if pos + T > self.cfg.max_seq_len:
                raise ValueError(f"gpt2 positions exceed max_seq_len ({pos + T} > {self.cfg.max_seq_len})")
            x = x + self.pos_emb(torch.arange(pos, pos + T, device=idx.device))[None]
        return self.drop(x)

    def _logits(self, h):
        h = self.norm(h)
        if self.cfg.arch == "shared":
            h = F.linear(h, self.emb_proj.weight.t())     # d -> E   (tied with the input projection)
        return F.linear(h, self.tok_emb.weight)            # tied output embedding

    def _decoder_layers(self):
        if self.cfg.arch == "shared":                       # cycle sharing: 0,1,..,5,0,1,..,5
            return [self.blocks[i % self.cfg.n_unique_layers] for i in range(self.cfg.n_layer)]
        return list(self.blocks)

    def encode(self, src):
        x = self._embed(src)
        for blk in self.enc_blocks:
            x = blk(x, rope=self._rope(), pos=0, causal=False)
        return self.enc_norm(x)

    # -- training forward -----------------------------------------------------------------------------------------
    def forward(self, idx, targets=None, src=None, loss_mask=None):
        """Decoder-only: idx (B,T) -> logits; loss vs targets (B,T).  encdec: src (B,S) is the encoder input."""
        memory = self.encode(src) if self.cfg.arch == "encdec" else None
        x = self._embed(idx)
        for blk in self._decoder_layers():
            x = blk(x, rope=self._rope(), pos=0, causal=True, memory=memory)
        logits = self._logits(x)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), reduction="none")
        if loss_mask is not None:
            m = loss_mask.reshape(-1).float()
            return logits, (loss * m).sum() / m.sum().clamp(min=1)
        return logits, loss.mean()

    # -- generation -----------------------------------------------------------------------------------------------
    @torch.no_grad()
    def generate(self, prompt, max_new_tokens, temperature=1.0, top_k=0, top_p=1.0, repetition_penalty=1.0,
                 no_repeat_ngram=0, banned_ids=(), bos_id=2, generator=None, use_cache=True):
        """prompt: (B, T) LongTensor, all rows the same length (no padding). Returns (B, max_new_tokens) new ids.
        temperature 0 = greedy."""
        B = prompt.shape[0]
        dev = prompt.device
        layers = self._decoder_layers()
        if self.cfg.arch == "encdec":
            memory = self.encode(prompt)
            seq = torch.full((B, 1), bos_id, dtype=torch.long, device=dev)   # decoder starts from <|bos|>
            history = prompt                                                   # for repetition penalties
        else:
            memory, seq, history = None, prompt, prompt
        caches = [dict() for _ in layers] if use_cache else None
        xcaches = [dict() for _ in layers] if (use_cache and memory is not None) else [None] * len(layers)
        out = []
        cur, pos = seq, 0
        for step in range(max_new_tokens):
            if use_cache:
                x = self._embed(cur, pos=pos)
                for li, blk in enumerate(layers):
                    x = blk(x, rope=self._rope(), pos=pos, causal=True, cache=caches[li], memory=memory,
                            xcache=xcaches[li])
                pos += cur.shape[1]
            else:
                x = self._embed(seq)
                for blk in layers:
                    x = blk(x, rope=self._rope(), pos=0, causal=True, memory=memory)
            logits = self._logits(x[:, -1:, :])[:, -1, :].float()
            full_hist = torch.cat([history] + out, dim=1) if out else history
            nxt = sample_next(logits, full_hist, temperature, top_k, top_p, repetition_penalty, no_repeat_ngram,
                              banned_ids, generator)
            out.append(nxt)
            seq = torch.cat([seq, nxt], dim=1)
            cur = nxt
        return torch.cat(out, dim=1)


def sample_next(logits, history, temperature, top_k, top_p, repetition_penalty, no_repeat_ngram, banned_ids,
                generator=None):
    """logits (B,V) float32, history (B,L) long (prompt + generated so far). Returns (B,1)."""
    logits = logits.clone()
    if banned_ids:
        logits[:, list(banned_ids)] = -float("inf")
    if repetition_penalty != 1.0:
        # CTRL-style penalty on tokens already present in the recent history (last 256 tokens)
        recent = history[:, -256:]
        sc = torch.gather(logits, 1, recent)
        sc = torch.where(sc > 0, sc / repetition_penalty, sc * repetition_penalty)
        logits.scatter_(1, recent, sc)
    if no_repeat_ngram and history.shape[1] >= no_repeat_ngram:
        n = no_repeat_ngram
        hist = history[:, -512:].tolist()
        for b, h in enumerate(hist):
            prefix = tuple(h[-(n - 1):]) if n > 1 else tuple()
            banned = {tuple(h[i:i + n])[-1] for i in range(len(h) - n + 1) if tuple(h[i:i + n - 1]) == prefix}
            if banned:
                logits[b, list(banned)] = -float("inf")
    if temperature <= 0:
        return logits.argmax(dim=-1, keepdim=True)
    logits = logits / temperature
    if top_k and top_k > 0:
        kth = torch.topk(logits, min(top_k, logits.shape[-1]), dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p < 1.0:
        sl, si = torch.sort(logits, descending=True, dim=-1)
        cp = torch.softmax(sl, dim=-1).cumsum(dim=-1)
        remove = cp - torch.softmax(sl, dim=-1) > top_p          # keep the smallest set with mass >= top_p
        sl = sl.masked_fill(remove, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, si, sl)
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, 1, generator=generator)


Writing /kaggle/working/model.py


In [3]:
%%writefile /kaggle/working/p5data.py
"""CS-781 Phase 5 (LSI): read train/dev/test once, normalise like our Phase-1 text, tokenise with OUR tokenizer,
and cache token ids (uint16) per split -> <out_dir>/<split>.npz.  Run once; training scripts only read the cache."""
import argparse, hashlib, json, os, re, sys, time, unicodedata
import numpy as np

# same punctuation map as Phase-1 pre-processing (copied from p2text.PUNCT)
PUNCT = {"‘": "'", "’": "'", "‚": "'", "‛": "'", "′": "'",
         "“": '"', "”": '"', "„": '"', "″": '"',
         "‐": "-", "‑": "-", "‒": "-", "–": "-", "—": "-",
         "―": "-", "−": "-", "•": "*", "·": "."}
_TABLE = str.maketrans(PUNCT)
_CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
_WS = re.compile(r"\s+")


def normalize(t):
    """Fast version of p2text.normalize_with_map (no character map needed here): fancy punctuation -> ASCII,
    NFKC, accents stripped, other non-ASCII dropped, control characters dropped, whitespace runs -> one space."""
    if not isinstance(t, str):
        t = " ".join(t) if isinstance(t, list) else str(t or "")
    t = t.translate(_TABLE)
    if not t.isascii():
        t = unicodedata.normalize("NFKC", t)
        t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode("ascii")
    t = _CTRL.sub("", t)
    return _WS.sub(" ", t).strip()


def load_records(path):
    d = json.load(open(path, encoding="utf-8"))
    if isinstance(d, dict):                       # tolerate {id: {...}} or {"data": [...]}
        if "data" in d and isinstance(d["data"], list):
            d = d["data"]
        else:
            d = [dict(v, id=k) if isinstance(v, dict) else {"id": k, "text": v} for k, v in d.items()]
    out = []
    for r in d:
        lab = r.get("labels", r.get("label", r.get("gold")))
        if lab is not None and not isinstance(lab, list):
            lab = [lab]
        out.append((str(r.get("id", r.get("ID", r.get("name")))), r.get("text", r.get("document", "")),
                    None if lab is None else sorted(set(int(x) for x in lab))))
    return out


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--raw_dir", required=True, help="folder with train.json, dev.json, test.json")
    p.add_argument("--tok", required=True)
    p.add_argument("--out_dir", required=True)
    a = p.parse_args(argv)
    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(a.tok)
    os.makedirs(a.out_dir, exist_ok=True)
    stats, hashes = {}, {}
    for split in ["train", "dev", "test"]:
        t0 = time.time()
        recs = load_records(os.path.join(a.raw_dir, f"{split}.json"))
        n_raw = len(recs)
        seen, keep, dup_same, dup_conflict = {}, [], 0, 0
        for did, text, lab in recs:                    # one entry per case id (the brief: one prediction per test id)
            if did in seen:
                if seen[did] == lab:
                    dup_same += 1
                else:
                    dup_conflict += 1
                continue
            seen[did] = lab
            keep.append((did, text, lab))
        del recs
        ids_all, offs, labs, loff, names, nchars, h = [], [0], [], [0], [], [], []
        B = 500
        for s_ in range(0, len(keep), B):
            part = keep[s_:s_ + B]
            texts = [normalize(t) for _, t, _ in part]
            encs = tok.encode_batch(texts, add_special_tokens=False)
            for (did, _, lab), txt, e in zip(part, texts, encs):
                arr = np.asarray(e.ids, dtype=np.uint16)
                ids_all.append(arr); offs.append(offs[-1] + len(arr))
                lab = lab or []
                labs.extend(lab); loff.append(loff[-1] + len(lab)); names.append(did); nchars.append(len(txt))
                h.append(hashlib.md5(txt.encode()).hexdigest())
        ids_cat = np.concatenate(ids_all) if ids_all else np.zeros(0, np.uint16)
        lens = np.diff(np.asarray(offs))
        np.savez(os.path.join(a.out_dir, f"{split}.npz"), ids=ids_cat, off=np.asarray(offs, np.int64),
                 lab=np.asarray(labs, np.int64), lab_off=np.asarray(loff, np.int64),
                 name=np.asarray(names), nchars=np.asarray(nchars, np.int64))
        hashes[split] = h
        nl = np.diff(np.asarray(loff))
        if split == "dev":          # local truth file in the format of the official evaluator
            json.dump([{"id": n, "gold": [int(x) for x in labs[loff[k]:loff[k + 1]]]} for k, n in enumerate(names)],
                      open(os.path.join(a.out_dir, "dev_truth.json"), "w"))
        freq = np.bincount(np.asarray(labs, np.int64), minlength=1) if labs else np.zeros(1, np.int64)
        st = {"records_in_file": n_raw, "unique_ids": len(keep), "duplicate_ids_same_label": dup_same,
              "duplicate_ids_conflicting_label": dup_conflict,
              "labels_per_doc_mean": float(nl.mean()) if split != "test" and len(nl) else None,
              "docs_without_labels": int((nl == 0).sum()) if split != "test" else None,
              "max_label_id": int(max(labs)) if labs else None, "label_freq": freq.tolist() if split != "test" else None,
              "tokens_mean": float(lens.mean()) if len(lens) else 0, "tokens_median": float(np.median(lens)) if len(lens) else 0,
              "tokens_p90": float(np.percentile(lens, 90)) if len(lens) else 0, "tokens_max": int(lens.max()) if len(lens) else 0,
              "frac_over_512": float((lens > 512).mean()) if len(lens) else 0,
              "frac_over_2048": float((lens > 2048).mean()) if len(lens) else 0,
              "empty_docs": int((lens == 0).sum()), "seconds": round(time.time() - t0, 1)}
        stats[split] = st
        print(split, json.dumps({k: v for k, v in st.items() if k != "label_freq"}), flush=True)
    tr = set(hashes["train"])
    stats["text_overlap"] = {"dev_in_train": sum(x in tr for x in hashes["dev"]),
                             "test_in_train": sum(x in tr for x in hashes["test"]),
                             "test_in_dev": len(set(hashes["test"]) & set(hashes["dev"]))}
    print("identical texts:", stats["text_overlap"], flush=True)
    L = 1 + max(stats["train"]["max_label_id"] or 0, stats["dev"]["max_label_id"] or 0)
    ftr = np.zeros(L, np.int64); fdv = np.zeros(L, np.int64)
    ftr[:len(stats["train"]["label_freq"])] = stats["train"]["label_freq"]; fdv[:len(stats["dev"]["label_freq"])] = stats["dev"]["label_freq"]
    stats["n_labels"] = L
    stats["labels_never_in_train"] = [int(i) for i in np.where(ftr == 0)[0]]
    stats["labels_in_dev_not_train"] = [int(i) for i in np.where((fdv > 0) & (ftr == 0))[0]]
    print(f"label space: {L} statute ids (0..{L - 1}) | train freq min {ftr.min()} median {int(np.median(ftr))} max {ftr.max()} | "
          f"ids with < 20 train cases: {int((ftr < 20).sum())} | never in train: {stats['labels_never_in_train']}", flush=True)
    json.dump(stats, open(os.path.join(a.out_dir, "stats.json"), "w"), indent=1)


if __name__ == "__main__":
    main()


Writing /kaggle/working/p5data.py


In [4]:
%%writefile /kaggle/working/p5ls.py
"""CS-781 Phase 5: Legal Statute Identification (multi-label) on our own Phase-1 models.

Model  = Phase-1 backbone (+ adaptation method) -> one vector per chunk (mean of token states)
         -> agg "attn": document vector by learned attention pooling over chunks -> L sigmoid outputs (one per statute)
         -> agg "max" : L sigmoid outputs per CHUNK, document score = max over its chunks (chunk-level prediction + aggregation)
Long documents ("rep"): last | first | hier_spread (K chunks of W tokens covering / spread over the whole document)
Methods: lorma (compulsory) | full | partial | adapter | prompt (soft prompt tuning) | head ; --kd_teacher adds distillation.
Loss: binary cross-entropy per statute (binary relevance) with optional per-statute positive weights, or focal loss.
Decision: thresholds chosen on dev (global vs per-statute), at least one statute per document (every case has >= 1).
Dev is used only for choosing the checkpoint and the thresholds, never for training.
"""
import argparse, json, math, os, random, sys, time, zipfile
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from model import LegalLM, ModelConfig                      # our Phase-1 model code (unchanged)


# ================================================================================================================
# adaptation modules (our own code)
# ================================================================================================================
class LoRMALinear(nn.Module):
    """LoRMA (Bihany, Patel & Modi, 2025), additive rank inflation: W' = (I + (alpha/r) B A) W0.
    y0 = W0 x ; y = y0 + s * B(A y0) (+ bias).  B = 0 -> starts exactly at W0."""

    def __init__(self, base: nn.Linear, r=16, alpha=32, dropout=0.05):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        d_out = base.out_features
        self.A = nn.Parameter(torch.randn(r, d_out) / math.sqrt(d_out))
        self.B = nn.Parameter(torch.zeros(d_out, r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        y0 = F.linear(x, self.base.weight)
        y = y0 + self.scale * F.linear(F.linear(self.drop(y0), self.A), self.B)
        return y if self.base.bias is None else y + self.base.bias


class AdapterOut(nn.Module):
    """Bottleneck adapter (Houlsby et al. 2019) on a sub-layer's output projection: y = W x ; y + Up(GELU(Down(y)))."""

    def __init__(self, base: nn.Linear, dim=64):
        super().__init__()
        self.base = base
        d = base.out_features
        self.a_down = nn.Linear(d, dim)
        self.a_up = nn.Linear(dim, d)
        nn.init.zeros_(self.a_up.weight); nn.init.zeros_(self.a_up.bias)

    def forward(self, x):
        y = self.base(x)
        return y + self.a_up(F.gelu(self.a_down(y)))


# ================================================================================================================
# the LSI network
# ================================================================================================================
class LSNet(nn.Module):
    def __init__(self, lm: LegalLM, n_labels, bidir=True, agg="attn", dropout=0.1):
        super().__init__()
        self.lm, self.arch, self.bidir, self.agg = lm, lm.cfg.arch, bidir, agg
        d = lm.cfg.d_model
        if self.arch == "encdec":                       # only the (bidirectional) encoder is used
            lm.blocks = nn.ModuleList()
        self.att = nn.Sequential(nn.Linear(d, 128), nn.Tanh(), nn.Linear(128, 1))   # learned pooling over chunks
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(d, n_labels)
        self.prompt = None                              # soft prompt (prompt tuning), set by setup_method

    def features(self, idx):
        lm = self.lm
        x = lm._embed(idx)
        P = 0
        if self.prompt is not None:                     # prompt tuning: P learned vectors in front of every chunk
            P = self.prompt.shape[0]
            x = torch.cat([self.prompt[None].expand(x.shape[0], -1, -1).to(x.dtype), x], 1)
        if self.arch == "encdec":
            for blk in lm.enc_blocks:
                x = blk(x, rope=lm._rope(), pos=0, causal=False)
            x = lm.enc_norm(x)
        else:
            for blk in lm._decoder_layers():
                x = blk(x, rope=lm._rope(), pos=0, causal=not self.bidir)
            x = lm.norm(x)
        return x[:, P:]

    def doc_logits_from_chunks(self, V, owner, n_docs):
        """V (N,d) chunk vectors, owner (N,) doc index -> (n_docs, L) logits."""
        if self.agg == "max":                           # chunk-level statute scores, document = max over chunks
            cl = self.out(self.drop(V)).float()
            out = torch.full((n_docs, cl.shape[1]), -1e4, device=V.device)
            return out.index_reduce_(0, owner, cl, "amax", include_self=True)
        s = self.att(V).squeeze(-1)
        s = s - s.max()
        w = torch.exp(s)
        den = torch.zeros(n_docs, device=V.device).index_add_(0, owner, w)
        doc = torch.zeros(n_docs, V.shape[1], device=V.device).index_add_(0, owner, w[:, None] * V)
        doc = doc / den.clamp(min=1e-12)[:, None]
        return self.out(self.drop(doc)).float()


def count_params(m):
    return sum(p.numel() for p in m.parameters()), sum(p.numel() for p in m.parameters() if p.requires_grad)


def setup_method(net, a):
    lm = net.lm
    for p in net.parameters():
        p.requires_grad = True
    m = a["method"]
    if m == "full":
        return
    for p in lm.parameters():
        p.requires_grad = False
    root = lm.enc_blocks if net.arch == "encdec" else lm.blocks
    if m == "lorma":
        tg = set(a["targets"].split(","))
        n = 0
        for mod in list(root.modules()):
            for name, ch in list(mod.named_children()):
                if isinstance(ch, nn.Linear) and name in tg:
                    setattr(mod, name, LoRMALinear(ch, a["rank"], a["alpha"], 0.05)); n += 1
        print(f"lorma: wrapped {n} linear layers", flush=True)
    elif m == "adapter":
        n = 0
        for mod in list(root.modules()):
            for name, ch in list(mod.named_children()):
                if isinstance(ch, nn.Linear) and name in ("o", "down"):
                    setattr(mod, name, AdapterOut(ch, a["adapter_dim"])); n += 1
        print(f"adapter: {n} bottleneck adapters (dim {a['adapter_dim']})", flush=True)
    elif m == "partial":
        blocks = list(root)
        top = blocks[-a["partial_layers"]:]
        for b in top:
            for p in b.parameters():
                p.requires_grad = True
        norm = lm.enc_norm if net.arch == "encdec" else lm.norm
        for p in norm.parameters():
            p.requires_grad = True
        print(f"partial: top {len(top)} of {len(blocks)} blocks + final norm trainable", flush=True)
    elif m == "prompt":
        E = lm.tok_emb.weight
        init_ids = torch.randint(min(1000, E.shape[0] // 4), E.shape[0], (a["prompt_len"],), generator=torch.Generator().manual_seed(0))
        with torch.no_grad():
            init = lm._embed(init_ids[None])[0].detach().clone()      # start from real token embeddings
        net.prompt = nn.Parameter(init)
        print(f"prompt tuning: {a['prompt_len']} soft prompt vectors (backbone frozen)", flush=True)
    elif m == "head":
        print("head: backbone frozen, only pooling + classifier train", flush=True)
    else:
        raise ValueError(m)


# ================================================================================================================
# data
# ================================================================================================================
class Split:
    def __init__(self, path, limit=0):
        z = np.load(path, allow_pickle=False)
        self.ids, self.off = z["ids"], z["off"]
        self.lab, self.lab_off = z["lab"], z["lab_off"]
        self.name = [str(x) for x in z["name"]]
        self.n = len(self.name) if not limit else min(limit, len(self.name))

    def doc(self, i):
        return self.ids[self.off[i]:self.off[i + 1]]

    def labels(self, i):
        return self.lab[self.lab_off[i]:self.lab_off[i + 1]]

    def multi_hot(self, idxs, L):
        Y = np.zeros((len(idxs), L), np.float32)
        for k, i in enumerate(idxs):
            ls = self.labels(i)
            ls = ls[ls < L]
            Y[k, ls] = 1.0
        return Y


def chunks_of(ids, rep, W, K):
    """-> list of np arrays (token ids) for one document. Every chunk has length W unless the whole doc is shorter."""
    n = len(ids)
    if n == 0:
        return [np.zeros(1, np.uint16)]
    if rep == "last":
        return [ids[-W:]]
    if rep == "first":
        return [ids[:W]]
    if n <= W:
        return [ids]
    if rep == "hier_spread":                         # K chunks covering the doc (contiguous if it fits, else spread)
        k = min(K, math.ceil(n / W))
        starts = np.linspace(0, n - W, k).round().astype(int)
        return [ids[s:s + W] for s in starts]
    raise ValueError(rep)


def doc_logits(net, chunk_lists, device, amp, max_chunks=32):
    flat = [(di, c) for di, cl in enumerate(chunk_lists) for c in cl]
    by_len = defaultdict(list)
    for j, (_, c) in enumerate(flat):
        by_len[len(c)].append(j)
    vecs = [None] * len(flat)
    for L, js in by_len.items():
        for s in range(0, len(js), max_chunks):
            part = js[s:s + max_chunks]
            x = torch.from_numpy(np.stack([flat[j][1] for j in part]).astype(np.int64)).to(device)
            with amp():
                h = net.features(x)
            v = h.float().mean(1)
            for k, j in enumerate(part):
                vecs[j] = v[k]
    V = torch.stack(vecs)
    owner = torch.tensor([di for di, _ in flat], device=device)
    return net.doc_logits_from_chunks(V, owner, len(chunk_lists))


@torch.no_grad()
def predict(net, split, idxs, a, device, amp):
    net.eval()
    out = []
    docs_per = max(1, a.eval_chunks // (1 if a.rep in ("last", "first") else a.chunks))
    for s in range(0, len(idxs), docs_per):
        part = idxs[s:s + docs_per]
        lg = doc_logits(net, [chunks_of(split.doc(i), a.rep, a.win, a.chunks) for i in part], device, amp, max_chunks=a.eval_chunks)
        out.append(torch.sigmoid(lg).cpu().numpy().astype(np.float32))
    net.train()
    return np.concatenate(out, 0) if out else np.zeros((0, net.out.out_features), np.float32)


# ================================================================================================================
# metrics (same definitions as the official evaluator: macro-F1 over all L statute ids, micro-F1, subset accuracy)
# ================================================================================================================
def scores(Y, P):
    """Y, P: (N, L) 0/1 arrays."""
    Y = Y.astype(bool); P = P.astype(bool)
    tp = (Y & P).sum(0).astype(float); fp = (~Y & P).sum(0).astype(float); fn = (Y & ~P).sum(0).astype(float)
    den = 2 * tp + fp + fn
    f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1e-12), 0.0)
    micro = 2 * tp.sum() / max(2 * tp.sum() + fp.sum() + fn.sum(), 1e-12)
    return {"macro_f1": float(f1.mean()), "micro_f1": float(micro), "subset_acc": float((Y == P).all(1).mean()) if len(Y) else 0.0,
            "per_label_f1": f1, "support": Y.sum(0).astype(int), "pred_count": P.sum(0).astype(int)}


GRID = np.round(np.arange(0.02, 0.951, 0.01), 2)


def decide(P, th, min_one=True):
    D = P >= th[None, :] if np.ndim(th) else P >= th
    if min_one:                                         # every case has at least one statute: keep the top one
        empty = ~D.any(1)
        if empty.any():
            D[np.where(empty)[0], P[empty].argmax(1)] = True
    return D


def tune_thresholds(Y, P, min_pos=5):
    """-> dict of strategies {name: (thresholds, scores)}: global@0.5, best global, per-statute (rare -> global)."""
    out = {}
    for mo in (False, True):
        out[f"global_0.5{'_min1' if mo else ''}"] = (np.float32(0.5), scores(Y, decide(P, 0.5, mo)))
    best_g = max(GRID, key=lambda t: (scores(Y, decide(P, t, False))["macro_f1"], -abs(t - 0.5)))
    out["global_best"] = (np.float32(best_g), scores(Y, decide(P, best_g, False)))
    out["global_best_min1"] = (np.float32(best_g), scores(Y, decide(P, best_g, True)))
    L = Y.shape[1]; th = np.full(L, best_g, np.float32)
    Yb = Y.astype(bool)
    for l in range(L):
        if Yb[:, l].sum() < min_pos:                    # too few dev examples: a per-statute threshold would overfit
            continue
        p = P[:, l][:, None] >= GRID[None, :]
        tp = (p & Yb[:, l][:, None]).sum(0); fp = (p & ~Yb[:, l][:, None]).sum(0); fn = (~p & Yb[:, l][:, None]).sum(0)
        f = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
        th[l] = GRID[int(np.argmax(f))]
    out["per_label"] = (th, scores(Y, decide(P, th, False)))
    out["per_label_min1"] = (th, scores(Y, decide(P, th, True)))
    return out


def preds_to_lists(D):
    return [[int(j) for j in np.where(row)[0]] for row in D]


# ================================================================================================================
def parse_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--run_name", required=True)
    p.add_argument("--base_run", required=True)
    p.add_argument("--base_file", default="best_model.pt")
    p.add_argument("--method", default="lorma", choices=["lorma", "full", "partial", "adapter", "prompt", "head"])
    p.add_argument("--kd_teacher", default="")
    p.add_argument("--kd_alpha", type=float, default=0.5, help="weight of the gold-label loss (1-alpha: teacher)")
    p.add_argument("--kd_T", type=float, default=2.0)
    p.add_argument("--save_train_probs", type=int, default=0)
    p.add_argument("--bidir", type=int, default=1)
    p.add_argument("--rep", default="hier_spread", choices=["last", "first", "hier_spread"])
    p.add_argument("--agg", default="attn", choices=["attn", "max"])
    p.add_argument("--win", type=int, default=512)
    p.add_argument("--chunks", type=int, default=4)
    p.add_argument("--bs", type=int, default=16, help="chunks per training step")
    p.add_argument("--eval_chunks", type=int, default=48)
    p.add_argument("--epochs", type=float, default=2.0)
    p.add_argument("--max_train_min", type=float, default=60.0)
    p.add_argument("--evals", type=int, default=4)
    p.add_argument("--dev_eval_limit", type=int, default=3000, help="dev cases used for checkpoint choice during training")
    p.add_argument("--lr", type=float, default=5e-5)
    p.add_argument("--peft_lr", type=float, default=5e-4)
    p.add_argument("--head_lr", type=float, default=2e-3)
    p.add_argument("--rank", type=int, default=16)
    p.add_argument("--alpha", type=float, default=32)
    p.add_argument("--targets", default="q,k,v,o")
    p.add_argument("--adapter_dim", type=int, default=64)
    p.add_argument("--partial_layers", type=int, default=4)
    p.add_argument("--prompt_len", type=int, default=20)
    p.add_argument("--loss", default="bce", choices=["bce", "focal"])
    p.add_argument("--pos_weight", default="sqrt", choices=["none", "sqrt"], help="per-statute positive weight sqrt(neg/pos)")
    p.add_argument("--wd", type=float, default=0.01)
    p.add_argument("--warmup", type=float, default=0.06)
    p.add_argument("--seed", type=int, default=13)
    p.add_argument("--init_run", default="", help="continue training from this Phase-5 run's best.pt (same method / reading)")
    p.add_argument("--data_dir", required=True)
    p.add_argument("--ckpt_dir", required=True)
    p.add_argument("--out_dir", required=True)
    p.add_argument("--hf_repo", default="")
    p.add_argument("--max_steps", type=int, default=-1)
    p.add_argument("--train_limit", type=int, default=0)
    p.add_argument("--dev_limit", type=int, default=0)
    return p.parse_args(argv)


def hf_up(repo, local, remote):
    if not repo:
        return
    from huggingface_hub import HfApi
    for k in range(3):
        try:
            HfApi(token=os.environ.get("HF_TOKEN")).upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=repo, repo_type="model")
            return
        except Exception as e:
            print("[hf] upload failed:", e, flush=True); time.sleep(15 * (k + 1))


def hf_get(repo, remote, local_dir):
    if not repo:
        return None
    from huggingface_hub import hf_hub_download
    try:
        return hf_hub_download(repo, remote, repo_type="model", local_dir=local_dir, token=os.environ.get("HF_TOKEN"))
    except Exception:
        return None


def main(argv=None):
    args = parse_args(argv)
    t_start = time.time()
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cuda = dev.type == "cuda"
    amp = (lambda: torch.autocast("cuda", dtype=torch.float16)) if cuda else (lambda: torch.autocast("cpu", enabled=False))
    run_dir = os.path.join(args.out_dir, args.run_name); os.makedirs(run_dir, exist_ok=True)
    logf = open(os.path.join(run_dir, "log.txt"), "a")

    def log(*m):
        s = f"[{time.strftime('%H:%M:%S')}][{args.run_name}] " + " ".join(str(x) for x in m)
        print(s, flush=True); logf.write(s + "\n"); logf.flush()

    # ---------------- data ----------------
    stats = json.load(open(os.path.join(args.data_dir, "stats.json")))
    L = int(stats["n_labels"])
    tr = Split(os.path.join(args.data_dir, "train.npz"), args.train_limit)
    dv = Split(os.path.join(args.data_dir, "dev.npz"), args.dev_limit)
    te = Split(os.path.join(args.data_dir, "test.npz"))
    tr_idx = [i for i in range(tr.n) if len(tr.labels(i)) > 0]
    dv_idx = list(range(dv.n)); te_idx = list(range(te.n))
    Ld = int(stats["dev"]["max_label_id"]) + 1 if stats["dev"].get("max_label_id") is not None else L
    Ld = min(Ld, L)   # official evaluator: label count = highest id in the truth + 1, and any predicted id >= that is INVALID
    Y_dv = dv.multi_hot(dv_idx, L)[:, :Ld]       # scored exactly like evaluate_phase5.py (on dev); ids >= Ld are never predicted
    ev_idx = dv_idx[:args.dev_eval_limit] if args.dev_eval_limit else dv_idx
    Y_ev = Y_dv[:len(ev_idx)]
    freq = np.zeros(L); cnt = np.bincount(np.concatenate([tr.labels(i) for i in tr_idx]), minlength=L)[:L]; freq[:len(cnt)] = cnt
    log(f"{L} statute ids (predicted / scored: 0..{Ld - 1}, as the official evaluator) | train {len(tr_idx)} docs ({freq.sum() / max(len(tr_idx), 1):.2f} statutes/doc) | dev {len(dv_idx)} "
        f"(checkpoint choice on {len(ev_idx)}) | test {len(te_idx)} | rep {args.rep} agg {args.agg} W {args.win} K {args.chunks}")

    # ---------------- model ----------------
    ck_path = os.path.join(args.ckpt_dir, "runs", args.base_run, args.base_file)
    if not os.path.exists(ck_path):
        hf_get(args.hf_repo, f"runs/{args.base_run}/{args.base_file}", args.ckpt_dir)
    ck = torch.load(ck_path, map_location="cpu", weights_only=False)
    lm_cfg = ck["config"]
    if lm_cfg["arch"] == "gpt2":                                  # learned positions: never exceed the trained length
        args.win = min(args.win, lm_cfg["max_seq_len"] - (args.prompt_len if args.method == "prompt" else 0))
    lm = LegalLM(ModelConfig(**lm_cfg))
    missing, _ = lm.load_state_dict(ck["model"], strict=False)
    bad = [k for k in missing if not k.startswith("blocks.")] if lm_cfg["arch"] == "encdec" else list(missing)
    assert not bad, f"missing backbone weights: {bad[:5]}"
    del ck
    net = LSNet(lm, L, bidir=bool(args.bidir), agg=args.agg)
    with torch.no_grad():                                           # start the classifier at the statute base rates
        net.out.bias.copy_(torch.tensor(np.log((freq + 1) / (len(tr_idx) - freq + 1)), dtype=torch.float32))
    setup_method(net, vars(args))
    if args.init_run:                                               # continue from an earlier run's best checkpoint
        ip = os.path.join(args.out_dir, args.init_run, "best.pt")
        if not os.path.exists(ip):
            ip = hf_get(args.hf_repo, f"phase5/runs/{args.init_run}/best.pt", os.path.join(args.out_dir, "_hub"))
        assert ip and os.path.exists(ip), f"init_run {args.init_run}: best.pt not found"
        sd0 = torch.load(ip, map_location="cpu", weights_only=False)["state_dict"]
        cur = net.state_dict()
        assert set(sd0) == set(cur), f"init_run {args.init_run}: parameters do not match this setup"
        net.load_state_dict({k: v.to(cur[k].dtype) for k, v in sd0.items()})
        print(f"continuing from {args.init_run}/best.pt ({len(sd0)} tensors)", flush=True)
    net.to(dev)
    total, trainable = count_params(net)
    log(f"backbone {args.base_run} ({lm_cfg['arch']}) | method {args.method}{' + KD from ' + args.kd_teacher if args.kd_teacher else ''}"
        f" | loss {args.loss} pos_weight {args.pos_weight} | params {total / 1e6:.2f}M trainable {trainable / 1e6:.3f}M ({100 * trainable / total:.2f}%)")

    teacher = None
    if args.kd_teacher:
        tp = os.path.join(args.out_dir, args.kd_teacher, "train_probs.npz")
        if not os.path.exists(tp):
            q = hf_get(args.hf_repo, f"phase5/runs/{args.kd_teacher}/train_probs.npz", os.path.join(args.out_dir, "_hub"))
            assert q is not None, f"teacher {args.kd_teacher}: train_probs.npz not found"
            tp = q
        z = np.load(tp)
        tmap = {str(n): k for k, n in enumerate(z["name"])}
        teacher = z["p"][[tmap[tr.name[i]] for i in tr_idx]].astype(np.float32)
        log(f"KD teacher {args.kd_teacher}: {teacher.shape} train probabilities, alpha {args.kd_alpha} T {args.kd_T}")

    # ---------------- optimiser / loss ----------------
    head_params = list(net.att.parameters()) + list(net.out.parameters())
    hp = set(id(p) for p in head_params)
    other = [p for p in net.parameters() if p.requires_grad and id(p) not in hp]
    groups = [{"params": other, "lr": args.lr if args.method in ("full", "partial") else args.peft_lr},
              {"params": head_params, "lr": args.head_lr}]
    groups = [g for g in groups if g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=args.wd)
    base_lrs = [g["lr"] for g in groups]
    scaler = torch.amp.GradScaler("cuda", enabled=cuda)
    pw = None
    if args.pos_weight == "sqrt":
        pw = torch.tensor(np.clip(np.sqrt((len(tr_idx) - freq) / np.maximum(freq, 1)), 1.0, 30.0), dtype=torch.float32, device=dev)
    log("positive weights:", None if pw is None else f"min {pw.min().item():.2f} median {pw.median().item():.2f} max {pw.max().item():.2f}")

    def loss_fn(logits, y):
        if args.loss == "focal":                                   # focal loss (gamma 2): focuses on hard / rare positives
            pr = torch.sigmoid(logits)
            ce = F.binary_cross_entropy_with_logits(logits, y, reduction="none", pos_weight=pw)
            pt = torch.where(y > 0.5, pr, 1 - pr)
            return (((1 - pt) ** 2) * ce).sum(1).mean()
        return F.binary_cross_entropy_with_logits(logits, y, reduction="none", pos_weight=pw).sum(1).mean()

    single = args.rep in ("last", "first")
    docs_per_step = max(1, args.bs // (1 if single else args.chunks))
    steps_per_epoch = math.ceil(len(tr_idx) / docs_per_step)
    total_steps = max(1, int(steps_per_epoch * args.epochs))
    if args.max_steps > 0:
        total_steps = min(total_steps, args.max_steps)
    warm = max(1, int(args.warmup * total_steps))
    budget = args.max_train_min * 60
    eval_every = max(1, total_steps // max(1, args.evals))
    log(f"{docs_per_step} docs/step | {steps_per_epoch} steps/epoch | planned {total_steps} steps | budget {args.max_train_min:.0f} min")

    rng = random.Random(args.seed)
    order, best = [], {"f1": -1.0}
    best_state, step, t_train, eval_time = None, 0, time.time(), 0.0
    if cuda:
        torch.cuda.reset_peak_memory_stats()
    trainable_names = [n for n, p in net.named_parameters() if p.requires_grad]
    pos = {i: k for k, i in enumerate(tr_idx)}

    def evaluate():
        nonlocal best_state, eval_time
        te0 = time.time()
        P = predict(net, dv, ev_idx, args, dev, amp)[:, :Ld]
        g = max(GRID[::3], key=lambda t: scores(Y_ev, decide(P, t, True))["macro_f1"])
        r = scores(Y_ev, decide(P, g, True))
        eval_time += time.time() - te0
        log(f"step {step}/{total_steps} | dev[{len(ev_idx)}] macro-F1 {r['macro_f1']:.4f} micro-F1 {r['micro_f1']:.4f} "
            f"(global threshold {g:.2f}) | {(time.time() - t_train) / 60:.1f} min")
        if r["macro_f1"] > best["f1"]:
            best.update(f1=r["macro_f1"], step=step)
            sd = net.state_dict()
            best_state = {k: sd[k].detach().to("cpu", copy=True) for k in trainable_names}

    stop = False
    net.train()
    while not stop:
        if not order:
            order = tr_idx[:]; rng.shuffle(order)
        batch = order[:docs_per_step]; order = order[docs_per_step:]
        lr_mult = min(1.0, (step + 1) / warm) * (0.5 * (1 + math.cos(math.pi * min(1.0, step / total_steps))))
        for g_, b in zip(opt.param_groups, base_lrs):
            g_["lr"] = b * max(lr_mult, 0.02)
        y = torch.from_numpy(tr.multi_hot(batch, L)).to(dev)
        logits = doc_logits(net, [chunks_of(tr.doc(i), args.rep, args.win, args.chunks) for i in batch], dev, amp, max_chunks=max(args.bs, 1))
        loss = loss_fn(logits, y)
        if teacher is not None:                                     # sigmoid KD: match the teacher's tempered probabilities
            T = args.kd_T
            pt = torch.from_numpy(teacher[[pos[i] for i in batch]]).to(dev).clamp(1e-5, 1 - 1e-5)
            qt = torch.sigmoid(torch.log(pt / (1 - pt)) / T)
            kd = F.binary_cross_entropy_with_logits(logits / T, qt, reduction="none").sum(1).mean() * T * T
            loss = args.kd_alpha * loss + (1 - args.kd_alpha) * kd
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_([p for g_ in groups for p in g_["params"]], 1.0)
        scaler.step(opt); scaler.update()
        step += 1
        if step == 30 and args.max_steps <= 0:
            sec = (time.time() - t_train) / step
            fit = int(max(60, 0.85 * budget / sec))
            if fit < total_steps:
                log(f"{sec:.2f} s/step -> time budget allows {fit} of {total_steps} steps; schedule shortened")
                total_steps = fit; eval_every = max(1, total_steps // max(1, args.evals))
            else:
                log(f"{sec:.2f} s/step -> planned {total_steps} steps fit the budget")
        if step % 100 == 0:
            log(f"step {step}/{total_steps} loss {loss.item():.4f} | {(time.time() - t_train) / 60:.1f} min")
        if step % eval_every == 0 or step >= total_steps:
            evaluate()
        if step >= total_steps or (time.time() - t_train) > budget:
            if step % eval_every != 0 and step < total_steps:
                evaluate()
            stop = True
    train_min = (time.time() - t_train) / 60
    log(f"training done: {step} steps in {train_min:.1f} min (dev evaluation {eval_time / 60:.1f} min) | best step {best['step']}")

    # ---------------- best checkpoint -> full dev, thresholds, test (/ train) ----------------
    sd = net.state_dict(); sd.update(best_state); net.load_state_dict(sd)
    t0 = time.time()
    P_dv = predict(net, dv, dv_idx, args, dev, amp)[:, :Ld]
    strat = tune_thresholds(Y_dv, P_dv)
    choice = max(strat, key=lambda k: strat[k][1]["macro_f1"])
    th, r = strat[choice]
    P_te = predict(net, te, te_idx, args, dev, amp)[:, :Ld]
    test_s = time.time() - t0
    D_te = decide(P_te, th, choice.endswith("min1"))
    preds = [{"id": te.name[i], "pred": lst} for i, lst in zip(te_idx, preds_to_lists(D_te))]
    assert len(preds) == te.n and len(set(x["id"] for x in preds)) == te.n
    assert all(len(set(x["pred"])) == len(x["pred"]) and all(type(v) is int and 0 <= v < Ld for v in x["pred"]) for x in preds)
    json.dump(preds, open(os.path.join(run_dir, "prediction.json"), "w"))
    with zipfile.ZipFile(os.path.join(run_dir, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(os.path.join(run_dir, "prediction.json"), arcname="prediction.json")
    D_dv = decide(P_dv, th, choice.endswith("min1"))
    json.dump([{"id": dv.name[i], "pred": lst} for i, lst in zip(dv_idx, preds_to_lists(D_dv))],
              open(os.path.join(run_dir, "dev_prediction.json"), "w"))
    np.savez_compressed(os.path.join(run_dir, "probs.npz"), dev_name=np.array([dv.name[i] for i in dv_idx]), dev_p=P_dv.astype(np.float16),
                        test_name=np.array([te.name[i] for i in te_idx]), test_p=P_te.astype(np.float16))
    json.dump({"strategy": choice, "thresholds": np.atleast_1d(th).astype(float).tolist()}, open(os.path.join(run_dir, "thresholds.json"), "w"))
    if args.save_train_probs:
        t1 = time.time()
        P_tr = predict(net, tr, tr_idx, args, dev, amp)
        np.savez_compressed(os.path.join(run_dir, "train_probs.npz"), name=np.array([tr.name[i] for i in tr_idx]), p=P_tr.astype(np.float16))
        log(f"train probabilities for KD saved ({(time.time() - t1) / 60:.1f} min)")
    torch.save({"state_dict": {k: v.half() if v.is_floating_point() else v for k, v in net.state_dict().items()},
                "args": vars(args), "lm_config": lm_cfg}, os.path.join(run_dir, "best.pt"))
    freq_dv = Y_dv.sum(0)
    fq = freq[:Ld]
    rare = fq < np.percentile(fq[fq > 0], 33) if (fq > 0).any() else np.zeros(Ld, bool)
    summary = {"run": args.run_name, "base_run": args.base_run, "arch": lm_cfg["arch"], "method": args.method,
               "kd_teacher": args.kd_teacher, "rep": args.rep, "agg": args.agg, "win": args.win, "chunks": args.chunks,
               "loss": args.loss, "pos_weight": args.pos_weight, "params_total": total, "params_trainable": trainable,
               "n_labels": Ld, "n_labels_model": L, "dev_macro_f1": r["macro_f1"], "dev_micro_f1": r["micro_f1"], "dev_subset_acc": r["subset_acc"],
               "threshold_strategy": choice,
               "dev_by_strategy": {k: {"macro_f1": v[1]["macro_f1"], "micro_f1": v[1]["micro_f1"], "subset_acc": v[1]["subset_acc"]}
                                   for k, v in strat.items()},
               "dev_macro_f1_rare_third": float(r["per_label_f1"][rare].mean()) if rare.any() else None,
               "dev_macro_f1_frequent": float(r["per_label_f1"][~rare].mean()) if (~rare).any() else None,
               "dev_per_label_f1": r["per_label_f1"].round(4).tolist(), "dev_support": freq_dv.astype(int).tolist(),
               "train_freq": freq[:Ld].astype(int).tolist(), "best_step": best["step"], "steps": step,
               "train_minutes": round(train_min, 1), "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if cuda else 0.0,
               "infer_dev_test_seconds": round(test_s, 1), "n_test": te.n,
               "test_labels_per_doc": float(D_te.sum(1).mean()), "dev_labels_per_doc_pred": float(D_dv.sum(1).mean()),
               "dev_labels_per_doc_gold": float(Y_dv.sum(1).mean()), "args": vars(args)}
    json.dump(summary, open(os.path.join(run_dir, "summary.json"), "w"), indent=1)
    log("DEV by threshold strategy: " + " | ".join(f"{k} {v[1]['macro_f1']:.4f}" for k, v in strat.items()))
    log(f"DEV macro-F1 {r['macro_f1']:.4f} micro-F1 {r['micro_f1']:.4f} subset-acc {r['subset_acc']:.4f} ({choice}) | "
        f"rare-third macro-F1 {summary['dev_macro_f1_rare_third']}")
    log(f"TEST: {te.n} docs, {D_te.sum(1).mean():.2f} statutes/doc -> {run_dir}/submission.zip")
    if args.hf_repo:
        files = ["prediction.json", "submission.zip", "dev_prediction.json", "probs.npz", "thresholds.json", "log.txt", "best.pt"]
        if args.save_train_probs:
            files.append("train_probs.npz")
        for fn in files:
            hf_up(args.hf_repo, os.path.join(run_dir, fn), f"phase5/runs/{args.run_name}/{fn}")
        hf_up(args.hf_repo, os.path.join(run_dir, "summary.json"), f"phase5/runs/{args.run_name}/summary.json")
    log(f"DONE in {(time.time() - t_start) / 60:.1f} min")


if __name__ == "__main__":
    main()


Writing /kaggle/working/p5ls.py


In [5]:
%%writefile /kaggle/working/p5ens.py
"""CS-781 Phase 5: ensembles of our fine-tuned LSI models (all built on our own Phase-1 models).
Each run saved P(statute | document) for every dev and test case (probs.npz). An ensemble = plain average of the members'
probabilities; thresholds (global or per statute) and the at-least-one-statute rule are chosen on dev, exactly like a single model.
  ens_top3 / ens_top5 : the K runs with the best dev macro-F1     ens_greedy : greedy forward selection on dev
"""
import argparse, json, os, shutil, zipfile
import numpy as np
import p5ls as M


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--data_dir", required=True)
    p.add_argument("--out_dir", required=True)
    p.add_argument("--hf_repo", default="")
    p.add_argument("--ks", default="3,5")
    p.add_argument("--min_dev", type=float, default=0.0)
    p.add_argument("--max_members", type=int, default=5)
    p.add_argument("--min_gain", type=float, default=0.002)
    a = p.parse_args(argv)
    L = int(json.load(open(os.path.join(a.data_dir, "stats.json")))["n_labels"])
    dv = M.Split(os.path.join(a.data_dir, "dev.npz")); te = M.Split(os.path.join(a.data_dir, "test.npz"))
    dev_names, test_names = dv.name[:dv.n], te.name[:te.n]
    st_ = json.load(open(os.path.join(a.data_dir, "stats.json")))
    Ld = min(L, int(st_["dev"]["max_label_id"]) + 1)        # same label space as evaluate_phase5.py on dev
    Y = dv.multi_hot(list(range(dv.n)), L)[:, :Ld]
    names = set(x for x in os.listdir(a.out_dir) if os.path.isdir(os.path.join(a.out_dir, x)))
    if a.hf_repo:
        from huggingface_hub import HfApi
        for f in HfApi(token=os.environ.get("HF_TOKEN")).list_repo_files(a.hf_repo):
            if f.startswith("phase5/runs/") and f.endswith("/summary.json"):
                names.add(f.split("/")[2])
    P, summ = {}, {}
    for n in sorted(names):
        if n.startswith("ens_") or n.startswith("_"):
            continue
        d = os.path.join(a.out_dir, n); os.makedirs(d, exist_ok=True)
        for fn in ["summary.json", "probs.npz"]:
            if not os.path.exists(os.path.join(d, fn)):
                q = M.hf_get(a.hf_repo, f"phase5/runs/{n}/{fn}", os.path.join(a.out_dir, "_hub"))
                if q:
                    shutil.copy(q, os.path.join(d, fn))
        if not (os.path.exists(os.path.join(d, "summary.json")) and os.path.exists(os.path.join(d, "probs.npz"))):
            continue
        s = json.load(open(os.path.join(d, "summary.json")))
        if "method" not in s or s["dev_macro_f1"] < a.min_dev:
            continue
        z = np.load(os.path.join(d, "probs.npz"))
        dm = {str(k): i for i, k in enumerate(z["dev_name"])}; tm = {str(k): i for i, k in enumerate(z["test_name"])}
        if not all(k in dm for k in dev_names) or not all(k in tm for k in test_names):
            print("skip (ids do not match):", n); continue
        P[n] = (z["dev_p"][[dm[k] for k in dev_names]][:, :Ld].astype(np.float32), z["test_p"][[tm[k] for k in test_names]][:, :Ld].astype(np.float32))
        summ[n] = s
    cands = sorted(P, key=lambda n: -summ[n]["dev_macro_f1"])
    print(f"[ensemble] {len(cands)} candidates:", ", ".join(f"{n}={summ[n]['dev_macro_f1']:.4f}" for n in cands), flush=True)
    if len(cands) < 2:
        print("[ensemble] fewer than 2 candidates"); return

    def best_of(members):
        pd = np.mean([P[m][0] for m in members], 0)
        st = M.tune_thresholds(Y, pd)
        k = max(st, key=lambda x: st[x][1]["macro_f1"])
        return k, st[k]

    def quick(members):                                   # fast score for the greedy search: best global threshold, min-one
        pd = np.mean([P[m][0] for m in members], 0)
        return max(M.scores(Y, M.decide(pd, t, True))["macro_f1"] for t in M.GRID[::3])

    def write(name, members):
        k, (th, r) = best_of(members)
        pt = np.mean([P[m][1] for m in members], 0)
        D = M.decide(pt, th, k.endswith("min1"))
        rd = os.path.join(a.out_dir, name); os.makedirs(rd, exist_ok=True)
        preds = [{"id": n, "pred": lst} for n, lst in zip(test_names, M.preds_to_lists(D))]
        assert len(preds) == len(set(test_names))
        assert all(len(set(x["pred"])) == len(x["pred"]) and all(type(v) is int and 0 <= v < Ld for v in x["pred"]) for x in preds)
        json.dump(preds, open(os.path.join(rd, "prediction.json"), "w"))
        with zipfile.ZipFile(os.path.join(rd, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as zf:
            zf.write(os.path.join(rd, "prediction.json"), arcname="prediction.json")
        Ddv = M.decide(np.mean([P[m][0] for m in members], 0), th, k.endswith("min1"))
        json.dump([{"id": n, "pred": lst} for n, lst in zip(dev_names, M.preds_to_lists(Ddv))], open(os.path.join(rd, "dev_prediction.json"), "w"))
        info = {"members": members, "threshold_strategy": k, "dev_macro_f1": r["macro_f1"], "dev_micro_f1": r["micro_f1"],
                "dev_subset_acc": r["subset_acc"], "test_labels_per_doc": float(D.sum(1).mean())}
        json.dump(info, open(os.path.join(rd, "summary.json"), "w"), indent=1)
        print(f"[ensemble] {name}: {len(members)} models {members} | dev macro-F1 {r['macro_f1']:.4f} micro-F1 {r['micro_f1']:.4f} "
              f"subset-acc {r['subset_acc']:.4f} ({k}) -> {rd}/submission.zip", flush=True)
        if a.hf_repo:
            for fn in ["prediction.json", "submission.zip", "dev_prediction.json", "summary.json"]:
                M.hf_up(a.hf_repo, os.path.join(rd, fn), f"phase5/runs/{name}/{fn}")

    for K in [int(k) for k in a.ks.split(",") if k.strip()]:
        if K <= len(cands):
            write(f"ens_top{K}", cands[:K])
    chosen = [cands[0]]; cur = quick(chosen)
    while len(chosen) < a.max_members:
        bn, bf = None, cur
        for n in cands:
            if n not in chosen:
                f = quick(chosen + [n])
                if f > bf:
                    bn, bf = n, f
        if bn is None or bf - cur < a.min_gain:
            break
        chosen.append(bn); cur = bf
    if len(chosen) > 1:
        write("ens_greedy", chosen)


if __name__ == "__main__":
    main()


Writing /kaggle/working/p5ens.py


In [6]:
%%writefile /kaggle/working/evaluate_phase5.py
#!/usr/bin/env python3
"""Local evaluator for Phase 5.

Truth JSON: a list of objects, [{"id":"...", "gold":[0,2,5,...]}, ...]
Prediction JSON: [{"id":"...", "pred":[0,2,5,...]}, ...]
"""
import argparse,json,sys
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def read_truth(path):
    try:
        x=json.load(open(path,encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")
    if not isinstance(x,list) or not x: raise ValueError('Truth must be a non-empty JSON list.')
    out={}; max_label=-1
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'gold' not in r: raise ValueError(f'Truth item {n} must contain id and gold.')
        if not isinstance(r['id'],str) or not r['id'].strip() or not isinstance(r['gold'],list): raise ValueError(f'Invalid truth item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate truth ID: {r["id"]}')
        seen=set()
        for label in r['gold']:
            if isinstance(label,bool) or not isinstance(label,int) or label<0 or label in seen: raise ValueError(f'Invalid/duplicate label in truth item {n}.')
            seen.add(label); max_label=max(max_label,label)
        out[r['id']]=r['gold']
    return out,max_label+1

def read_pred(path,nlab):
    x=json.load(open(path,encoding='utf-8'))
    if not isinstance(x,list) or not x: raise ValueError('Prediction must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'pred' not in r: raise ValueError(f'Prediction item {n} must contain id and pred.')
        if not isinstance(r['id'],str) or not isinstance(r['pred'],list) or r['id'] in out: raise ValueError(f'Invalid or duplicate prediction item {n}.')
        seen=set()
        for x in r['pred']:
            if isinstance(x,bool) or not isinstance(x,int) or x<0 or x>=nlab or x in seen: raise ValueError(f'Invalid/duplicate label in prediction item {n}.')
            seen.add(x)
        out[r['id']]=r['pred']
    return out

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 5 multilabel predictions on your own held-out split.')
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True); a=ap.parse_args()
    t,n=read_truth(a.truth); p=read_pred(a.prediction,n)
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    yt=np.zeros((len(t),n),dtype=np.uint8); yp=np.zeros((len(t),n),dtype=np.uint8)
    for i,k in enumerate(t):
        for x in t[k]: yt[i,x]=1
        for x in p[k]: yp[i,x]=1
    print(f'Examples: {len(t)}'); print(f'Labels: {n}')
    print(f'Macro-F1: {f1_score(yt,yp,average="macro",zero_division=0):.6f}')
    micro=f1_score(yt,yp,average='micro',zero_division=0)
    subset=accuracy_score(yt,yp)
    print(f'Micro-F1: {micro:.6f}')
    print(f'Subset Accuracy: {subset:.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)


Writing /kaggle/working/evaluate_phase5.py


In [7]:
# ===================== P5 CELL 3 - Data: normalise + tokenise ONCE with our tokenizer (cached on Hugging Face) =====================
# First session: downloads the official Phase_5_LSI files (~480 MB), tokenises them (~10 min) and uploads the cache.
sys.path.insert(0, WORK)
hub_files = set(api.list_repo_files(CKPT_REPO))
cache = ["train.npz", "dev.npz", "test.npz", "stats.json", "dev_truth.json"]
if all(f"phase5/data/{f}" in hub_files for f in cache):
    for f in cache:
        shutil.copy(hf_hub_download(CKPT_REPO, f"phase5/data/{f}", repo_type="model", token=HF_TOKEN, local_dir=f"{P5}/hub"), f"{DATA}/{f}")
    print("token cache downloaded from Hugging Face")
else:
    t0 = time.time()
    for p in [x.path for x in api.list_repo_tree(SRC_REPO, repo_type="dataset", path_in_repo="Phase_5_LSI", recursive=True)
              if getattr(x, "size", None) is not None]:
        lp = hf_hub_download(SRC_REPO, p, repo_type="dataset", token=HF_TOKEN, local_dir=f"{P5}/raw")
        shutil.copy(lp, f"{RAW}/{p.split('/')[-1]}")
    missing = [n for n in ["train.json", "dev.json", "test.json"] if not os.path.exists(f"{RAW}/{n}")]
    assert not missing, f"missing files {missing} - paste this output to Claude"
    print("Phase-5 files:", {f: f"{os.path.getsize(f'{RAW}/{f}') / 1e6:.0f} MB" for f in sorted(os.listdir(RAW))}, f"| {time.time() - t0:.0f}s")
    r = subprocess.run([sys.executable, f"{WORK}/p5data.py", "--raw_dir", RAW, "--tok", TOK_PATH, "--out_dir", DATA], capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    assert r.returncode == 0, "data preparation failed - paste this output to Claude"
    for f in cache:
        api.upload_file(path_or_fileobj=f"{DATA}/{f}", path_in_repo=f"phase5/data/{f}", repo_id=CKPT_REPO, repo_type="model")
    print(f"tokenised + uploaded in {(time.time() - t0) / 60:.1f} min")
    shutil.rmtree(f"{P5}/raw", ignore_errors=True); shutil.rmtree(RAW, ignore_errors=True)
stats = json.load(open(f"{DATA}/stats.json"))
for sp in ["train", "dev", "test"]:
    s = stats[sp]
    print(f"{sp:5s}: {s['unique_ids']:6d} docs (records {s['records_in_file']}, duplicate ids {s['duplicate_ids_same_label']}+{s['duplicate_ids_conflicting_label']} conflicting)"
          f" | statutes/doc {s['labels_per_doc_mean']} | no statute {s['docs_without_labels']} | tokens mean {s['tokens_mean']:.0f} median {s['tokens_median']:.0f}"
          f" p90 {s['tokens_p90']:.0f} max {s['tokens_max']} | >512: {100 * s['frac_over_512']:.0f}% >2048: {100 * s['frac_over_2048']:.0f}% | empty {s['empty_docs']}")
import numpy as np
ftr = np.array(stats["train"]["label_freq"] + [0] * (stats["n_labels"] - len(stats["train"]["label_freq"])))
print(f"label space: {stats['n_labels']} statute ids | train cases per statute: min {ftr.min()} median {int(np.median(ftr))} max {ftr.max()} | "
      f"statutes with < 20 / < 100 train cases: {(ftr < 20).sum()} / {(ftr < 100).sum()}")
print("statute ids never in train:", stats["labels_never_in_train"], "| in dev but not train:", stats["labels_in_dev_not_train"])
print("identical texts across splits:", stats["text_overlap"])


phase5/data/train.npz: reconstructing file:   0%|          |  0.00B /  150MB            

phase5/data/train.npz: downloading bytes:           |  0.00B            

phase5/data/dev.npz: reconstructing file:   0%|          |  0.00B / 39.7MB            

phase5/data/dev.npz: downloading bytes:           |  0.00B            

phase5/data/test.npz: reconstructing file:   0%|          |  0.00B / 49.0MB            

phase5/data/test.npz: downloading bytes:           |  0.00B            

stats.json:   0%|          | 0.00/3.27k [00:00<?, ?B/s]

dev_truth.json:   0%|          | 0.00/460k [00:00<?, ?B/s]

token cache downloaded from Hugging Face
train:  42750 docs (records 42750, duplicate ids 0+0 conflicting) | statutes/doc 3.704982456140351 | no statute 0 | tokens mean 1702 median 644 p90 4157 max 140464 | >512: 57% >2048: 25% | empty 0
dev  :  10181 docs (records 10181, duplicate ids 0+0 conflicting) | statutes/doc 3.9157253707887243 | no statute 0 | tokens mean 1902 median 806 p90 4478 max 115772 | >512: 62% >2048: 29% | empty 0
test :  13019 docs (records 13019, duplicate ids 0+0 conflicting) | statutes/doc None | no statute None | tokens mean 1850 median 799 p90 4273 max 219377 | >512: 62% >2048: 27% | empty 0
label space: 100 statute ids | train cases per statute: min 272 median 679 max 11730 | statutes with < 20 / < 100 train cases: 0 / 0
statute ids never in train: [] | in dev but not train: []
identical texts across splits: {'dev_in_train': 118, 'test_in_train': 150, 'test_in_dev': 61}


In [8]:
# ===================== P5 CELL 7 - Run the experiments on both GPUs (queue with KD dependency, resumable) =====================
# Every run has a TIME BUDGET (max_train_min): it trains until the budget or the planned epochs, evaluating on dev
# several times and keeping the best checkpoint. Finished runs (phase5/runs/<run>/summary.json on HF) are skipped.
import threading
CK = "/tmp/cs781/ckpt"
OUT = f"{WORK}/p5runs"; os.makedirs(OUT, exist_ok=True)
for run, fn in [("dapt_llama", "best_model.pt"), ("explore_gpt2", "final_model.pt"),
                ("explore_encdec", "final_model.pt"), ("explore_shared", "final_model.pt")]:
    hf_hub_download(CKPT_REPO, f"runs/{run}/{fn}", repo_type="model", local_dir=CK, token=HF_TOKEN)
print("Phase-1 checkpoints ready")

COMMON = dict(win=512, chunks=4, bs=16, epochs=2, evals=4, dev_eval_limit=3000, lr=5e-5, peft_lr=5e-4, head_lr=2e-3,
              rank=16, alpha=32, rep="hier_spread", agg="attn", loss="bce", pos_weight="sqrt", max_train_min=55)
LLAMA  = dict(base_run="dapt_llama", base_file="best_model.pt")
GPT2   = dict(base_run="explore_gpt2", base_file="final_model.pt", bidir=0)        # GPT-2: left-to-right (Phase 2 lesson)
ENCDEC = dict(base_run="explore_encdec", base_file="final_model.pt")
SHARED = dict(base_run="explore_shared", base_file="final_model.pt")
RUNS = [   # priority order. Default reading: up to 4 chunks of 512 tokens covering the whole document (statutes can be anywhere)
    ("ls_llama_partial_hier",  {**LLAMA,  "method": "partial", "partial_layers": 4, "save_train_probs": 1}),   # best Phase-4 recipe, KD teacher
    ("ls_llama_lorma_hier",    {**LLAMA,  "method": "lorma"}),                                                 # LoRMA (compulsory)
    ("ls_shared_kd_hier",      {**SHARED, "method": "full", "kd_teacher": "ls_llama_partial_hier", "max_train_min": 45}),  # KD
    ("ls_gpt2_lorma_hier",     {**GPT2,   "method": "lorma", "max_train_min": 45}),
    ("ls_encdec_lorma_hier",   {**ENCDEC, "method": "lorma", "max_train_min": 40}),
    ("ls_llama_full_hier",     {**LLAMA,  "method": "full", "lr": 3e-5}),                                      # full fine-tuning
    ("ls_llama_adapter_hier",  {**LLAMA,  "method": "adapter", "peft_lr": 1e-3, "max_train_min": 45}),         # adapters
    ("ls_llama_partial_max",   {**LLAMA,  "method": "partial", "partial_layers": 4, "agg": "max"}),             # chunk-level prediction + max
    ("ls_llama_partial_last",  {**LLAMA,  "method": "partial", "partial_layers": 4, "rep": "last", "max_train_min": 25}),   # section study
    ("ls_llama_partial_first", {**LLAMA,  "method": "partial", "partial_layers": 4, "rep": "first", "max_train_min": 25}),  # section study
    ("ls_llama_prompt_hier",   {**LLAMA,  "method": "prompt", "prompt_len": 20, "peft_lr": 1e-2, "max_train_min": 35}),    # prompt tuning
    ("ls_shared_full_hier",    {**SHARED, "method": "full", "max_train_min": 45}),                             # same student WITHOUT KD
    ("ls_llama_partial_focal", {**LLAMA,  "method": "partial", "partial_layers": 4, "loss": "focal", "pos_weight": "none", "max_train_min": 50}),  # loss study
    ("ls_llama_partial_nopw",  {**LLAMA,  "method": "partial", "partial_layers": 4, "pos_weight": "none", "max_train_min": 40}),   # class weighting study
]
CONT = dict(epochs=2, max_train_min=135, warmup=0.02, evals=5)
RUNS2 = [  # ---- round 2 (runs FIRST): the best round-1 runs had seen the data only about once -> continue their training ----
    ("r2_llama_partial_hier_cont", {**LLAMA, "method": "partial", "partial_layers": 4, **CONT, "init_run": "ls_llama_partial_hier"}),
    ("r2_llama_partial_max_cont",  {**LLAMA, "method": "partial", "partial_layers": 4, "agg": "max", **CONT, "init_run": "ls_llama_partial_max"}),
]
RUNS = RUNS2 + RUNS
ONLY = None          # e.g. ["cj_llama_lorma_last"] to run a subset

def p5_cmd(name, cfg, out_dir, hf_repo, extra=()):
    cmd = [sys.executable, f"{WORK}/p5ls.py", "--run_name", name, "--data_dir", DATA, "--ckpt_dir", CK,
           "--out_dir", out_dir, "--hf_repo", hf_repo]
    for k, v in {**COMMON, **cfg}.items():
        cmd += [f"--{k}", str(v)]
    return cmd + list(extra)

# ---- safety test: 3 training steps + dev + FULL test prediction + zip on the REAL data; nothing uploaded ----
t_s = time.time()
smoke = p5_cmd("_smoke", {**SHARED, "method": "lorma", "max_steps": 3, "evals": 1, "dev_eval_limit": 64},
               f"{P5}/smoke", "", ["--train_limit", "64", "--dev_limit", "64"])
r = subprocess.run(smoke, env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"), capture_output=True, text=True)
print("\n".join([l for l in (r.stdout + r.stderr).splitlines() if "][_smoke]" in l][-8:]))
if r.returncode != 0 or not os.path.exists(f"{P5}/smoke/_smoke/submission.zip"):
    print("\n".join((r.stdout + r.stderr).splitlines()[-30:]))
    raise RuntimeError("SAFETY TEST FAILED - nothing was trained. Send this output to Claude.")
print(f"safety test passed in {time.time() - t_s:.0f}s\n")
if torch.cuda.device_count() == 0:
    print("CPU session: switch Accelerator to GPU T4 x2 and run again."); RUNS = []

hub_files = set(api.list_repo_files(CKPT_REPO))
done, failed, running, lock = set(), set(), set(), threading.Lock()
pending = [(n, c) for n, c in RUNS if ONLY is None or n in ONLY]
for n, c in list(pending):
    if f"phase5/runs/{n}/summary.json" in hub_files:          # finished in an earlier session -> skip
        os.makedirs(f"{OUT}/{n}", exist_ok=True)
        for fn in ["summary.json", "submission.zip", "prediction.json", "dev_prediction.json", "probs.npz", "thresholds.json"]:
            if f"phase5/runs/{n}/{fn}" in hub_files:
                p = hf_hub_download(CKPT_REPO, f"phase5/runs/{n}/{fn}", repo_type="model", local_dir=f"{P5}/hub", token=HF_TOKEN)
                shutil.copy(p, f"{OUT}/{n}/{fn}")
        done.add(n); pending.remove((n, c)); print("already done:", n)

def next_job():
    with lock:
        for i, (n, c) in enumerate(pending):
            t = c.get("kd_teacher")
            if t and t not in done and f"phase5/runs/{t}/train_probs.npz" in hub_files:
                done.add(t)
            if t and t not in done and t not in running and all(t != m for m, _ in pending):
                failed.add(t)
            if t and t in failed:
                pending.pop(i); failed.add(n); print(f"SKIP {n}: teacher {t} failed / not available", flush=True); return "AGAIN"
            if not t or t in done:
                running.add(n); return pending.pop(i)
        return "WAIT" if pending else None

def worker(gpu):
    while True:
        job = next_job()
        if job is None: return
        if job in ("WAIT", "AGAIN"):
            time.sleep(30 if job == "WAIT" else 0); continue
        name, cfg = job
        with open(f"{OUT}/{name}.out", "w") as f:
            r = subprocess.run(p5_cmd(name, cfg, OUT, CKPT_REPO), env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)),
                               stdout=f, stderr=subprocess.STDOUT)
        ok_ = r.returncode == 0 and os.path.exists(f"{OUT}/{name}/summary.json")
        with lock:
            (done if ok_ else failed).add(name); running.discard(name)
            print(f"{'FINISHED' if ok_ else 'FAILED  '} {name} on GPU {gpu}", flush=True)
            if not ok_:
                print(f"===== last lines of {name}.out =====\n" + "\n".join(open(f"{OUT}/{name}.out").read().splitlines()[-30:]), flush=True)

def status_line(name):
    p = f"{OUT}/{name}.out"
    if not os.path.exists(p): return "(waiting)"
    keep = [l for l in open(p).read().splitlines() if "step" in l or "DEV" in l or "Error" in l or "DONE" in l]
    return keep[-1][:170] if keep else "(starting)"

threads = [threading.Thread(target=worker, args=(g,)) for g in range(max(1, torch.cuda.device_count()))]
for t in threads: t.start()
t0 = time.time()
while any(t.is_alive() for t in threads):
    for t in threads: t.join(timeout=300 / len(threads))
    print(f"--- {(time.time() - t0) / 60:.0f} min | done {sorted(done)} | failed {sorted(failed)}", flush=True)
    for n, _ in RUNS:
        if n not in done and n not in failed and (ONLY is None or n in ONLY): print(f"   {n:26s} {status_line(n)}", flush=True)
print("ALL FINISHED. done:", sorted(done), "| failed:", sorted(failed))


runs/dapt_llama/best_model.pt: reconstructing file:   0%|          |  0.00B /  431MB            

runs/dapt_llama/best_model.pt: downloading bytes:           |  0.00B            

runs/explore_gpt2/final_model.pt: reconstructing file:   0%|          |  0.00B /  444MB            

runs/explore_gpt2/final_model.pt: downloading bytes:           |  0.00B            

runs/explore_encdec/final_model.pt: reconstructing file:   0%|          |  0.00B /  469MB            

runs/explore_encdec/final_model.pt: downloading bytes:           |  0.00B            

... (125 outputs left out)

phase5/runs/ls_llama_partial_nopw/probs.(…): reconstructing file:   0%|          |  0.00B / 4.42MB            

phase5/runs/ls_llama_partial_nopw/probs.(…): downloading bytes:           |  0.00B            

thresholds.json:   0%|          | 0.00/2.03k [00:00<?, ?B/s]

already done: ls_llama_partial_nopw
--- 5 min | done ['ls_encdec_lorma_hier', 'ls_gpt2_lorma_hier', 'ls_llama_adapter_hier', 'ls_llama_full_hier', 'ls_llama_lorma_hier', 'ls_llama_partial_first', 'ls_llama_partial_focal', 'ls_llama_partial_hier', 'ls_llama_partial_last', 'ls_llama_partial_max', 'ls_llama_partial_nopw', 'ls_llama_prompt_hier', 'ls_shared_full_hier', 'ls_shared_kd_hier'] | failed []
   r2_llama_partial_hier_cont [13:20:40][r2_llama_partial_hier_cont] step 1500/21376 loss 24.9687 | 4.6 min
   r2_llama_partial_max_cont  [13:20:54][r2_llama_partial_max_cont] step 1700/21376 loss 27.8255 | 4.8 min
--- 10 min | done ['ls_encdec_lorma_hier', 'ls_gpt2_lorma_hier', 'ls_llama_adapter_hier', 'ls_llama_full_hier', 'ls_llama_lorma_hier', 'ls_llama_partial_first', 'ls_llama_partial_focal', 'ls_llama_partial_hier', 'ls_llama_partial_last', 'ls_llama_partial_max', 'ls_llama_partial_nopw', 'ls_llama_prompt_hier', 'ls_shared_full_hier', 'ls_shared_kd_hier'] | failed []
   r2_llama_partia

In [9]:
# ===================== P5 CELL 8 - Results, threshold strategies, rare statutes, co-occurrence, confusions, zips =====================
import numpy as np, itertools
ZIPS = f"{WORK}/zips"; os.makedirs(ZIPS, exist_ok=True)
rows = []
for n, _ in RUNS if RUNS else []:
    p = f"{OUT}/{n}/summary.json"
    if not os.path.exists(p): continue
    rows.append(json.load(open(p)))
    if os.path.exists(f"{OUT}/{n}/submission.zip"):
        shutil.copy(f"{OUT}/{n}/submission.zip", f"{ZIPS}/{n}.zip")
rows.sort(key=lambda s: -s["dev_macro_f1"])
print(f"{'run':24s} {'model':6s} {'method':8s} {'reading':12s} {'loss':9s} {'trainable':>9s} {'macroF1':>7s} {'microF1':>7s} {'subset':>6s} "
      f"{'rare':>5s} {'freq':>5s} {'steps':>9s} {'train':>6s} {'GPU':>5s} {'inf s':>5s} {'lab/doc':>7s}  threshold strategy")
for s in rows:
    m = s["method"] + ("+KD" if s.get("kd_teacher") else "")
    rd = s["rep"] + ("/max" if s["agg"] == "max" else "")
    ls = s["loss"] + ("+pw" if s["pos_weight"] != "none" else "")
    print(f"{s['run']:24s} {s['arch']:6s} {m:8s} {rd:12s} {ls:9s} {s['params_trainable'] / 1e6:8.2f}M {s['dev_macro_f1']:7.4f} {s['dev_micro_f1']:7.4f} "
          f"{s['dev_subset_acc']:6.3f} {s['dev_macro_f1_rare_third'] or 0:5.3f} {s['dev_macro_f1_frequent'] or 0:5.3f} {s['best_step']:>4d}/{s['steps']:<4d} "
          f"{s['train_minutes']:5.1f}m {s['peak_gpu_gb']:5.2f} {s['infer_dev_test_seconds']:5.0f} {s['dev_labels_per_doc_pred']:4.2f}/{s['dev_labels_per_doc_gold']:.2f}  {s['threshold_strategy']}")
print("\nmacroF1 / microF1 / subset = official metrics on the FULL dev set with the chosen threshold strategy; rare = macro-F1 over the"
      " rarest third of statutes (by train count), freq = the other two thirds; lab/doc = predicted / gold statutes per dev document")
print("\nthreshold strategies on dev (macro-F1):  global 0.5 | global 0.5 + at-least-one | best global | best global + one | per-statute | per-statute + one")
for s in rows:
    d = s["dev_by_strategy"]
    print(f"   {s['run']:24s} " + " | ".join(f"{d[k]['macro_f1']:.4f}" for k in ["global_0.5", "global_0.5_min1", "global_best", "global_best_min1", "per_label", "per_label_min1"]))
if rows:
    b = rows[0]; L = b["n_labels"]
    f1 = np.array(b["dev_per_label_f1"]); sup = np.array(b["dev_support"]); trf = np.array(b["train_freq"])
    print(f"\nbest single run: {b['run']}")
    print("   statutes MOST MISSED (lowest dev F1 among those with >= 10 dev cases):  id: F1 (dev cases, train cases)")
    for l in [l for l in np.argsort(f1) if sup[l] >= 10][:10]:
        print(f"      {l:3d}: {f1[l]:.3f} ({sup[l]}, {trf[l]})")
    for lo, hi in [(0, 50), (50, 200), (200, 1000), (1000, 10 ** 9)]:
        m = (trf >= lo) & (trf < hi)
        if m.any(): print(f"   statutes with {lo}-{hi if hi < 10 ** 9 else 'max'} train cases: {m.sum():3d} statutes, mean dev F1 {f1[m].mean():.3f}")
    gold = {r["id"]: set(r["gold"]) for r in json.load(open(f"{DATA}/dev_truth.json"))}
    pred = {r["id"]: set(r["pred"]) for r in json.load(open(f"{OUT}/{b['run']}/dev_prediction.json"))}
    co_g, co_p, conf = collections.Counter(), collections.Counter(), collections.Counter()
    for k, g in gold.items():
        p_ = pred.get(k, set())
        co_g.update(itertools.combinations(sorted(g), 2)); co_p.update(itertools.combinations(sorted(p_), 2))
        for miss in g - p_:
            for extra in p_ - g:
                conf[(miss, extra)] += 1
    print("   most frequent statute PAIRS in dev gold (pair: gold count / predicted count):",
          ", ".join(f"{a}+{c}: {n}/{co_p[(a, c)]}" for (a, c), n in co_g.most_common(8)))
    print("   most common CONFUSIONS (gold statute missed -> other statute predicted instead, same document):",
          ", ".join(f"{a}->{c}: {n}" for (a, c), n in conf.most_common(8)))
print(f"\nlabel space used: statute ids 0..{stats['n_labels'] - 1} in the data; predictions are limited to ids below the highest dev id + 1 "
      f"({rows[0]['n_labels'] if rows else '?'}) because the official evaluator rejects any id >= (highest id in its truth file) + 1")
if os.path.exists(f"{WORK}/evaluate_phase5.py"):     # cross-check with the OFFICIAL evaluator
    print("\nofficial evaluate_phase5.py on dev (truth = dev labels, prediction = each run's dev_prediction.json):")
    for s in rows:
        dp = f"{OUT}/{s['run']}/dev_prediction.json"
        r = subprocess.run([sys.executable, f"{WORK}/evaluate_phase5.py", "--truth", f"{DATA}/dev_truth.json", "--prediction", dp],
                           capture_output=True, text=True)
        print(f"   {s['run']:24s} " + " | ".join(l.strip() for l in (r.stdout + r.stderr).splitlines()))
json.dump(rows, open(f"{P5}/p5_results.json", "w"), indent=1)
api.upload_file(path_or_fileobj=f"{P5}/p5_results.json", path_in_repo="phase5/results/p5_results.json", repo_id=CKPT_REPO, repo_type="model")
print("\nresults table uploaded to Hugging Face: phase5/results/p5_results.json")
print("zips ready in", ZIPS, ":", sorted(os.listdir(ZIPS)))


run                      model  method   reading      loss      trainable macroF1 microF1 subset  rare  freq     steps  train   GPU inf s lab/doc  threshold strategy
r2_llama_partial_hier_cont llama  partial  hier_spread  bce+pw       27.71M  0.4209  0.5110  0.052 0.337 0.462 21376/21376  72.9m  2.38   608 4.71/3.92  per_label
r2_llama_partial_max_cont llama  partial  hier_spread/max bce+pw       27.71M  0.4167  0.5069  0.051 0.334 0.457 21375/21376  72.0m  2.38   605 4.87/3.92  per_label_min1
ls_llama_partial_hier    llama  partial  hier_spread  bce+pw       27.71M  0.4007  0.4870  0.037 0.319 0.441 14308/14310  46.7m  2.38   558 5.16/3.92  per_label
ls_llama_partial_max     llama  partial  hier_spread/max bce+pw       27.71M  0.4004  0.4878  0.039 0.319 0.441 14871/14871  48.0m  2.38   559 5.05/3.92  per_label
ls_llama_full_hier       llama  full     hier_spread  bce+pw      107.94M  0.3939  0.4814  0.034 0.309 0.436 7750/7750  50.2m  6.45   568 5.06/3.92  per_label
ls_llama_partial_

In [10]:
# ===================== P5 CELL 9 - Ensembles (plain top-K averages + greedy), thresholds chosen on dev =====================
r = subprocess.run([sys.executable, f"{WORK}/p5ens.py", "--data_dir", DATA, "--out_dir", OUT, "--hf_repo", CKPT_REPO], capture_output=True, text=True)
print("\n".join([l for l in (r.stdout + r.stderr).splitlines() if "[ensemble]" in l or "Error" in l or "Traceback" in l][-20:]))
if r.returncode != 0:
    print("\n".join((r.stdout + r.stderr).splitlines()[-30:]))
for o in sorted(os.listdir(OUT)):
    if o.startswith("ens_") and os.path.exists(f"{OUT}/{o}/submission.zip"):
        shutil.copy(f"{OUT}/{o}/submission.zip", f"{ZIPS}/{o}.zip")
if os.path.exists(f"{WORK}/evaluate_phase5.py"):
    for o in sorted(x for x in os.listdir(OUT) if x.startswith("ens_")):
        dp = f"{OUT}/{o}/dev_prediction.json"
        if os.path.exists(dp):
            r = subprocess.run([sys.executable, f"{WORK}/evaluate_phase5.py", "--truth", f"{DATA}/dev_truth.json", "--prediction", dp], capture_output=True, text=True)
            print(f"official evaluator, {o}: " + " | ".join(l.strip() for l in (r.stdout + r.stderr).splitlines()))
print("zips:", sorted(os.listdir(ZIPS)))


[ensemble] 16 candidates: r2_llama_partial_hier_cont=0.4209, r2_llama_partial_max_cont=0.4167, ls_llama_partial_hier=0.4007, ls_llama_partial_max=0.4004, ls_llama_full_hier=0.3939, ls_llama_partial_focal=0.3930, ls_llama_partial_nopw=0.3926, ls_llama_adapter_hier=0.3718, ls_llama_partial_last=0.3678, ls_llama_partial_first=0.3657, ls_llama_lorma_hier=0.3575, ls_shared_kd_hier=0.3379, ls_llama_prompt_hier=0.3150, ls_shared_full_hier=0.3043, ls_gpt2_lorma_hier=0.2871, ls_encdec_lorma_hier=0.2679
[ensemble] ens_top3: 3 models ['r2_llama_partial_hier_cont', 'r2_llama_partial_max_cont', 'ls_llama_partial_hier'] | dev macro-F1 0.4295 micro-F1 0.5192 subset-acc 0.0559 (per_label) -> /kaggle/working/p5runs/ens_top3/submission.zip
[ensemble] ens_top5: 5 models ['r2_llama_partial_hier_cont', 'r2_llama_partial_max_cont', 'ls_llama_partial_hier', 'ls_llama_partial_max', 'ls_llama_full_hier'] | dev macro-F1 0.4263 micro-F1 0.5131 subset-acc 0.0469 (per_label) -> /kaggle/working/p5runs/ens_top5/subm